In [ ]:
import plotly.graph_objects as go
import numpy as np

def generate_curvature_sim():
    # Parámetros
    R_sphere = 2.0  # Radio de la esfera (curvatura 1/R)
    size = 0.5      # Tamaño del parche de área (d_theta)

    # Crear figura
    fig = go.Figure()

    # Función para crear un parche de superficie
    def get_sphere_patch(dn):
        r = R_sphere + dn
        phi = np.linspace(np.pi/4, np.pi/4 + size, 20)
        theta = np.linspace(0, size, 20)
        PHI, THETA = np.meshgrid(phi, theta)
        X = r * np.sin(PHI) * np.cos(THETA)
        Y = r * np.sin(PHI) * np.sin(THETA)
        Z = r * np.cos(PHI)
        return X, Y, Z

    def get_plane_patch(dn):
        # El plano no cambia su área con dn
        x = np.linspace(-size, size, 20)
        y = np.linspace(-size, size, 20)
        X, Y = np.meshgrid(x, y)
        Z = np.full_like(X, dn + R_sphere) # Lo situamos a la misma altura para comparar
        return X, Y, Z

    # Añadir trazas iniciales (dn = 0)
    xs, ys, zs = get_sphere_patch(0)
    fig.add_trace(go.Surface(x=xs, y=ys, z=zs, colorscale='Reds', showscale=False, name='Esfera (Curva)'))

    xp, yp, zp = get_plane_patch(0)
    fig.add_trace(go.Surface(x=xp, y=yp, z=zp, colorscale='Blues', showscale=False, name='Plano (Recto)'))

    # Crear los frames para la animación (variando dn)
    frames = []
    dns = np.linspace(0, 1.5, 30)
    for dn in dns:
        xs, ys, zs = get_sphere_patch(dn)
        xp, yp, zp = get_plane_patch(dn)
        frames.append(go.Frame(data=[
            go.Surface(x=xs, y=ys, z=zs),
            go.Surface(x=xp, y=yp, z=zp)
        ], name=f"dn_{dn}"))

    fig.frames = frames

    # Configuración de layout y slider
    fig.update_layout(
        title="Expansión del Área: Esfera vs Plano (Efecto de la Curvatura)",
        scene=dict(
            xaxis=dict(range=[-3, 4]),
            yaxis=dict(range=[-3, 4]),
            zaxis=dict(range=[0, 5]),
            aspectmode='cube'
        ),
        updatemenus=[{
            "buttons": [{"args": [None, {"frame": {"duration": 50, "redraw": True}}],
                         "label": "Play", "method": "animate"}],
            "type": "buttons"
        }],
        sliders=[{
            "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "method": "animate"}],
                       "label": str(round(dns[i], 2)), "method": "animate"} for i, f in enumerate(frames)]
        }]
    )

    fig.show()

# Ejecutar
# generate_curvature_sim()

In [ ]:
generate_curvature_sim()

Principio Físico y Matemático BaseEl código compara cómo cambia el tamaño de una pequeña superficie (un parche de área) cuando nos desplazamos una distancia normal $dn$ perpendicular a la superficie original:En el Plano (Espacio Euclidiano / Plano):Las líneas normales (perpendiculares a la superficie) son paralelas entre sí. Al desplazarte $dn$, el área del parche se mantiene constante:$$dA(dn) = dA_0$$El crecimiento relativo del área respecto al desplazamiento es nulo.En la Esfera (Espacio Curvo / No Euclidiano):Las líneas normales divergen hacia afuera a partir del centro de la esfera. Conforme $r$ aumenta ($r = R + dn$), la superficie de la esfera crece proporcionalmente a $r^2$:$$dA(dn) = \left(1 + \frac{dn}{R}\right)^2 dA_0$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Definición de la malla (Grid)
# Ajustamos el rango para que rodee a las cargas en z = d y z = -d
d = 1.0
limit = 3.0
n_points = 40  # Aumentar para mayor suavidad, pero consume más memoria
x, y, z = np.mgrid[-limit:limit:n_points*1j,
                  -limit:limit:n_points*1j,
                  -limit:limit:n_points*1j]

# 2. Función de potencial (Basada en la imagen)
# Añadimos un pequeño epsilon para evitar la división por cero en las cargas
eps = 1e-6
r1 = np.sqrt(x**2 + y**2 + (z - d)**2) + eps
r2 = np.sqrt(x**2 + y**2 + (z + d)**2) + eps
potencial = (1/r1) - (1/r2)

# 3. Crear la figura interactiva
fig = go.Figure()

# Definimos los valores de la constante C que queremos explorar con el slider
c_values = np.linspace(-0.5, 0.5, 101)

# Añadimos una traza por cada valor de C (Isosuperficie)
for i, c in enumerate(c_values):
    fig.add_trace(
        go.Isosurface(
            x=x.flatten(),
            y=y.flatten(),
            z=z.flatten(),
            value=potencial.flatten(),
            isomin=c,
            isomax=c,
            surface_count=1,
            colorscale='Viridis',
            showscale=False,
            visible=(i == len(c_values)//2)  # Solo una visible al inicio
        )
    )

# 4. Configuración del Slider
steps = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)},
              {"title": f"Superficie de Nivel C = {c_values[i]:.2f}"}],
    )
    step["args"][0]["visible"][i] = True
    steps.append(step)

sliders = [dict(
    active=len(c_values)//2,
    currentvalue={"prefix": "Valor de la constante C: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    title=f"Superficie de Nivel C = {c_values[len(c_values)//2]:.2f}",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube'
    )
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Definir parámetros (puedes cambiarlos a tu gusto)
a = 2.0  # Dado que a = b

# 2. Crear la malla de puntos (u, v) extendiendo un poco más allá de [-a, a]
u = np.linspace(-1.5 * a, 1.5 * a, 200)
v = np.linspace(-1.5 * a, 1.5 * a, 200)
U, V = np.meshgrid(u, v)


# 3. Definir la función Lambda producto según el diagrama
def lambda_2d(u, v, a):
    # Lambda(u) = a - |u| si |u| <= a, de lo contrario 0
    lambda_u = np.where(np.abs(u) <= a, a - np.abs(u), 0)
    # Lambda(v) = a - |v| si |v| <= a, de lo contrario 0
    lambda_v = np.where(np.abs(v) <= a, a - np.abs(v), 0)
    return lambda_u * lambda_v


Z = lambda_2d(U, V, a)

# 4. Crear el gráfico de superficie en Plotly
fig = go.Figure(
    data=[
        go.Surface(
            x=U,
            y=V,
            z=Z,
            colorscale="Viridis",
            colorbar=dict(title="Λ(u)Λ(v)"),
        )
    ]
)

# 5. Configurar el diseño (Layout) de la escena
fig.update_layout(
    title=f"Gráfico 3D de Λ(u)Λ(v) con a = {a} (x_c = y_c = 0)",
    scene=dict(
        xaxis=dict(title="u (Eje X)"),
        yaxis=dict(title="v (Eje Y)"),
        zaxis=dict(title="Λ(u)Λ(v)"),
    ),
    margin=dict(l=65, r=50, b=65, t=90),
)

# Mostrar el gráfico interactivo
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Definir el dominio y los parámetros de los pulsos
t = np.linspace(-4, 4, 400)      # Eje x para los pulsos
tau_vals = np.linspace(-2.5, 2.5, 80) # Desplazamiento del pulso móvil (slider)
ancho_a = 1.0                    # Ancho de los pulsos cuadrados

# Pulso fijo f(t) centrado en 0
p1 = np.where(np.abs(t) <= ancho_a / 2, 1.0, 0.0)

# 2. Configurar la figura inicial (tau = tau_vals[0])
tau_0 = tau_vals[0]
p2_0 = np.where(np.abs(t - tau_0) <= ancho_a / 2, 1.0, 0.0)
traslape_0 = p1 * p2_0

fig = go.Figure(
    data=[
        go.Scatter(x=t, y=p1, mode='lines', name='Pulso Fijo f(t)', line=dict(color='#1f77b4', width=2)),
        go.Scatter(x=t, y=p2_0, mode='lines', name='Pulso Móvil g(t - τ)', line=dict(color='#ff7f0e', width=2)),
        go.Scatter(x=t, y=traslape_0, mode='lines', name='Área de Traslape', fill='tozeroy', line=dict(color='#2ca02c', width=0), opacity=0.4),
        go.Scatter(x=[tau_0], y=[0], mode='lines+markers', name='Correlación Resultante Λ(τ)', line=dict(color='#d62728', width=3), marker=dict(size=6))
    ],
    layout=go.Layout(
        title="<b>Correlación Cruzada en Tiempo Real: Pulsos Cuadrados → Función Triangular</b>",
        xaxis=dict(range=[-3.5, 3.5], title="<b>Eje Espacial / Temporal (t / τ)</b>"),
        yaxis=dict(range=[-0.1, 1.3], title="<b>Amplitud / Área</b>"),
        hovermode="x unified",
        template="plotly_white",
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            x=0.05, y=1.15,
            buttons=[
                dict(label="► Reproducir", method="animate", args=[None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True}]),
                dict(label="❚❚ Pausa", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
            ]
        )]
    )
)

# 3. Crear los cuadros (frames) para cada valor del slider
frames = []
sliders_steps = []

for idx, tau in enumerate(tau_vals):
    # Pulso desplazado g(t - tau)
    p2 = np.where(np.abs(t - tau) <= ancho_a / 2, 1.0, 0.0)
    traslape = p1 * p2

    # Traza acumulada de la función triangular hasta la posición tau
    taus_hasta_ahora = tau_vals[:idx+1]
    corr_hasta_ahora = np.where(np.abs(taus_hasta_ahora) <= ancho_a, ancho_a - np.abs(taus_hasta_ahora), 0.0)

    # Cuadro de animación
    frame = go.Frame(
        data=[
            go.Scatter(x=t, y=p1, mode='lines', name='Pulso Fijo f(t)'),
            go.Scatter(x=t, y=p2, mode='lines', name='Pulso Móvil g(t - τ)'),
            go.Scatter(x=t, y=traslape, mode='lines', fill='tozeroy'),
            go.Scatter(x=taus_hasta_ahora, y=corr_hasta_ahora, mode='lines+markers')
        ],
        name=f"frame_{idx}"
    )
    frames.append(frame)

    # Configuración del paso del slider
    slider_step = {
        "args": [
            [f"frame_{idx}"],
            {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}
        ],
        "label": f"{tau:.2f}",
        "method": "animate"
    }
    sliders_steps.append(slider_step)

fig.frames = frames

# 4. Añadir el control del Slider a la figura
fig.update_layout(
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14},
            "prefix": "Desplazamiento (τ) = ",
            "visible": True,
            "xanchor": "right"
        },
        "pad": {"b": 10, "t": 40},
        "len": 0.9,
        "x": 0.05,
        "y": -0.05,
        "steps": sliders_steps
    }]
)

fig.show()